# 05. itertools and cache

**Tier:** Intermediate. **Why you'd use this:** You need the first few values of an infinite counter, and a recursive function that repeats the same arguments. `islice` cuts a lazy stream. `functools.cache` remembers answers for hashable arguments.

**After this lesson you can:** slice a lazy `count`; wrap a function with `cache` by calling it; explain why unhashable arguments fail.
About 25 minutes. **Uses:** [Generators](../07-iteration/02-generators-lesson.ipynb), [Loops](../03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. Why is a heap after `heapify` not fully sorted?
2. How do you get a max from a min-heap of numbers?

<details><summary>Answers</summary>

Only the parent-child rule holds. Negate the numbers, or use `nlargest`.
</details>


## The idea

`itertools` tools are lazy: they hand out values as you pull. `count` never ends. `islice` pulls a finite window. `functools.cache` wraps a function so matching hashable arguments reuse the stored return value. Call `cache(raw)` to wrap. Decorators come later; the idea is the same.

<details><summary>Explain more</summary>

Without a slice, looping `count` forever never finishes. Caching trades memory for repeated work. Arguments must be hashable because they become dict keys inside the cache.
</details>


## See it work

Take four counts, then cache a recursive Fibonacci.


In [1]:
import functools
import itertools


def first_counts(n):
    # 1. Cut a finite window from a lazy stream.
    return list(itertools.islice(itertools.count(0), n))


def fib(n):
    def raw(value):
        if value < 2:
            return value
        return raw(value - 1) + raw(value - 2)

    # 2. Wrap by calling cache. Arguments must be hashable.
    cached = functools.cache(raw)
    return cached(n)


print(first_counts(4))
print(fib(8))


[0, 1, 2, 3]
21


## Check yourself

Answer these before you run the cell.

1. Does `islice(count(0), 3)` build the whole infinite sequence first?
2. Why does caching a function that takes a list as its argument fail?


In [2]:
print(list(itertools.islice(itertools.count(0), 3)))


def raw(value):
    return value


cached = functools.cache(raw)
print(cached(2))
print(cached(2))
print(cached.cache_info())


[0, 1, 2]
2
2
CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)


<details><summary>Answers</summary>

No. It pulls three values and stops. Lists are unhashable, so they cannot be cache keys.
</details>


## Why it works this way

`itertools` builders return iterators. They do not allocate the full result until a consumer pulls. `count` can be infinite because nothing materializes ahead of the pull. `islice` is that consumer with a length limit. This matches the generator lesson: laziness is about when work runs, not about a different answer. Chaining lazy tools keeps intermediate lists out of memory until you finally call `list` or loop. Deep dive: [The iteration protocol](../../internals/07-iteration/01-protocol-lesson.ipynb).

`functools.cache` stores a dict from argument tuples to return values. Hashable arguments are required because dict keys need hashes. A list argument raises `TypeError` on the first call. Hits skip the function body; misses run it and store the result. Memory grows with distinct argument combinations. `cache_info` reports hits and misses so you can see the win on a recursive function that would otherwise recompute the same subproblems.

Wrapping with `cached = cache(raw)` is the same transformation a decorator performs later. Learning it as a call keeps the mechanism visible: functions are values, and `cache` returns a new function that remembers. The original function stays pure if you want correct cache hits. Side effects inside a cached function run only on misses, which can surprise you.

Cost: `islice` of n values is proportional to n pulls. A cached recursive Fibonacci drops from exponential calls to about n distinct subproblems. That is the design reason to memoize. Bound lazy streams. Cache pure functions with hashable inputs. Materialize with `list` only when a later step needs reuse or random access.


In [3]:
stream = itertools.count(0)
print(list(itertools.islice(stream, 3)))
print(list(itertools.islice(stream, 3)))


def raw(value):
    return value * value


cached = functools.cache(raw)
print(cached(3), cached(3), cached.cache_info())


[0, 1, 2]
[3, 4, 5]
9 9 CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)


In [4]:
def raw(value):
    return value


cached = functools.cache(raw)
cached([1, 2])


TypeError: unhashable type: 'list'

## When to use it

Use `islice` to bound lazy streams. Use `cache` when a pure function repeats hashable arguments. Use a list when you need random access to the whole result. Don't cache functions with unhashable or mutable arguments. Don't loop `count` without a bound.


## Common mistake

Calling a cached function with a list argument and reading the `TypeError` as a bug in the function body. The cache key is the problem.


In [5]:
def total(values):
    return sum(values)


cached = functools.cache(total)
print(cached([1, 2, 3]))


TypeError: unhashable type: 'list'

Pass a tuple when the values are fixed. Tuples are hashable when their items are.


In [6]:
def total(values):
    return sum(values)


cached = functools.cache(total)
print(cached((1, 2, 3)))


6


## Try it

Change `fib` so it prints `cache_info` after the call. Which numbers show that later Fibonacci values reuse earlier ones?


## Recap

- `itertools` tools are lazy. `islice` pulls a finite window.
- `cache` memoizes hashable arguments.
- Wrap with `cache(fn)` for now; `@` comes later.

**Next:** [practice](05-itertools-practice.ipynb)
